# 00 - Status Quo Naloxone Distribution

This notebook derives the current ("status quo") naloxone kit distribution
across Rhode Island cities and towns, used as the baseline against which the
greedy algorithm (`06_greedy_benchmark.ipynb`) and metamodel-based optimization
(`05_find_optimal_solutions.ipynb`) are compared in `07_compare_results.ipynb`.

**Input:** `R_Files/Inputs/MasterTable.xlsx` — historical naloxone kit distribution data by year and city/town

**Output:** `results/outputs/status_quo_results.csv` — the status quo allocation and its projected 2026 overdose deaths

> **Note:** `R_Files/` is the third-party microsimulation model's own directory and isn't included in this public repo (see the root README). This notebook can't be re-run here for that reason — it's included as a read-through of the methodology, with its real output already saved to `results/outputs/`.

In [ ]:
from pathlib import Path

# Repo root (1 level up from notebooks/)
BASE_DIR = Path.cwd().parents[0]

# Build file path
file_path = BASE_DIR / "R_Files" / "Inputs" / "MasterTable.xlsx"

file_path

In [2]:
# Get the current naloxone kit values by city/town from the master data table

import pandas as pd

# Read Excel file
df = pd.read_excel(file_path, sheet_name = 'NxDataOEND')

# Show first few rows
print(df)


   year  Barrington  Bristol  Burrillville  Central Falls  Charlestown  \
0  2015           4       10            36              5            5   
1  2016          12       32           121             16           16   
2  2017           7       35            28             53           27   
3  2018          47      105            61             91           32   
4  2019         100      343            23             14           16   
5  2020          62       30           212             38           35   
6  2021          88      157            12            250          187   
7  2022         192      216            65            213          452   
8  2023         288      249           220            274          811   

   Coventry  Cranston  Cumberland  East Greenwich  ...  Scituate  Smithfield  \
0        19       144           6               7  ...         3           8   
1        63       477          19              22  ...        11          27   
2        50       2

In [3]:
# Keep only the last row of the DataFrame
df = df.iloc[[-1]].reset_index(drop=True)
print(df)

   year  Barrington  Bristol  Burrillville  Central Falls  Charlestown  \
0  2023         288      249           220            274          811   

   Coventry  Cranston  Cumberland  East Greenwich  ...  Scituate  Smithfield  \
0       130      3057         114             154  ...       103         450   

   South Kingstown  Tiverton  Warren  Warwick  West Greenwich  West Warwick  \
0             4243       535    2404     3736              64          1845   

   Westerly  Woonsocket  
0      1211        4107  

[1 rows x 40 columns]


In [4]:
df = df.iloc[:, 1:]
print(df)

   Barrington  Bristol  Burrillville  Central Falls  Charlestown  Coventry  \
0         288      249           220            274          811       130   

   Cranston  Cumberland  East Greenwich  East Providence  ...  Scituate  \
0      3057         114             154             2234  ...       103   

   Smithfield  South Kingstown  Tiverton  Warren  Warwick  West Greenwich  \
0         450             4243       535    2404     3736              64   

   West Warwick  Westerly  Woonsocket  
0          1845      1211        4107  

[1 rows x 39 columns]


In [5]:
row_sum = df.sum(axis=1)
print(row_sum)

coefficient = 50000/49677

df = df * coefficient
print(df)

0    49677
dtype: int64
   Barrington     Bristol  Burrillville  Central Falls  Charlestown  \
0  289.872577  250.618999    221.430441     275.781549   816.273124   

    Coventry     Cranston  Cumberland  East Greenwich  East Providence  ...  \
0  130.84526  3076.876623  114.741228      155.001308      2248.525475  ...   

     Scituate  Smithfield  South Kingstown    Tiverton       Warren  \
0  103.669706  452.925901      4270.587998  538.478572  2419.630815   

       Warwick  West Greenwich  West Warwick     Westerly   Woonsocket  
0  3760.291483       64.416128   1856.996195  1218.873926  4133.703726  

[1 rows x 39 columns]


In [6]:
import numpy as np

# Extract the row
row = df.iloc[0]

# Step 1: Round each value to the nearest integer (keeping decimal parts if possible)
rounded_row = np.round(row).astype(int)

# Step 2: Check if the sum is exactly 50,000
rounded_sum = rounded_row.sum()

# If the sum is not exactly 50,000, adjust the largest values
diff = 50000 - rounded_sum
if diff != 0:
    # Adjust the values to ensure the sum equals 50,000
    abs_diff = np.abs(row - np.floor(row))  # Compute absolute fractional difference
    max_diff_index = abs_diff.idxmax()  # Identify column with the largest fractional part

    # Adjust the row value at the index of the largest fractional part
    rounded_row[max_diff_index] += diff

# Replace the original row with the adjusted row
df.iloc[0] = rounded_row.astype(int)

# Output the result
print(df)

print(df.iloc[0].sum())

   Barrington  Bristol  Burrillville  Central Falls  Charlestown  Coventry  \
0       290.0    251.0         221.0          276.0        816.0     131.0   

   Cranston  Cumberland  East Greenwich  East Providence  ...  Scituate  \
0    3077.0       115.0           155.0           2249.0  ...     104.0   

   Smithfield  South Kingstown  Tiverton  Warren  Warwick  West Greenwich  \
0       453.0           4271.0     538.0  2420.0   3760.0            64.0   

   West Warwick  Westerly  Woonsocket  
0        1855.0    1219.0      4134.0  

[1 rows x 39 columns]
50000.0


In [7]:
# Duplicate the row 100 times
df_repeated = pd.concat([df] * 100, ignore_index=True)
print(df_repeated)

    Barrington  Bristol  Burrillville  Central Falls  Charlestown  Coventry  \
0        290.0    251.0         221.0          276.0        816.0     131.0   
1        290.0    251.0         221.0          276.0        816.0     131.0   
2        290.0    251.0         221.0          276.0        816.0     131.0   
3        290.0    251.0         221.0          276.0        816.0     131.0   
4        290.0    251.0         221.0          276.0        816.0     131.0   
..         ...      ...           ...            ...          ...       ...   
95       290.0    251.0         221.0          276.0        816.0     131.0   
96       290.0    251.0         221.0          276.0        816.0     131.0   
97       290.0    251.0         221.0          276.0        816.0     131.0   
98       290.0    251.0         221.0          276.0        816.0     131.0   
99       290.0    251.0         221.0          276.0        816.0     131.0   

    Cranston  Cumberland  East Greenwich  East Prov

In [ ]:
from pathlib import Path

# Repo root (1 level up from notebooks/)
BASE_DIR = Path.cwd().parents[0]

output_dir = BASE_DIR / "results" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

file_path = output_dir / "status_quo_distribution.csv"
file_path

In [9]:
# Export to a csv
df_repeated.to_csv(file_path)

## Get Projected Deaths

`status_quo_distribution.csv` was run through the actual simulation on the HPC
cluster over the 100 calibrated parameter seeds to produce
`status_quo_distribution_deaths.csv`, imported below.

In [ ]:
from pathlib import Path

# Repo root (1 level up from notebooks/)
BASE_DIR = Path.cwd().parents[0]

file_path = BASE_DIR / "results" / "outputs" / "status_quo_distribution_deaths.csv"
print(file_path)

In [11]:
# Import the final results:

# Read CSV file
new_df = pd.read_csv(file_path)
new_df = new_df.iloc[:, 1:]
print(new_df)

    od_death_2024  od_death_2025  od_death_2026  Barrington  Bristol  \
0             366            314            324         290      251   
1             376            358            362         290      251   
2             331            344            341         290      251   
3             316            326            342         290      251   
4             309            294            293         290      251   
..            ...            ...            ...         ...      ...   
95            379            390            416         290      251   
96            333            361            368         290      251   
97            383            327            339         290      251   
98            379            370            387         290      251   
99            326            322            297         290      251   

    Burrillville  Central.Falls  Charlestown  Coventry  Cranston  ...  \
0            221            276          816       131      30

In [12]:
average = new_df.iloc[:100, 2].mean()
print(f"Projected number of deaths from status quo distribution: {average}")

Projected number of deaths from status quo distribution: 362.93


In [13]:
# Save the results here to be used later

# Keep only the first row and all columns except the first 3
status_quo_results = new_df.iloc[[0], 3:].copy()

# Insert the new column at the beginning
status_quo_results.insert(0, "od_death_2026", average)

status_quo_results

,od_death_2026,Barrington,Bristol,Burrillville,Central.Falls,Charlestown,Coventry,Cranston,Cumberland,East.Greenwich,...,Scituate,Smithfield,South.Kingstown,Tiverton,Warren,Warwick,West.Greenwich,West.Warwick,Westerly,Woonsocket
0,362.93,290,251,221,276,816,131,3077,115,155,...,104,453,4271,538,2420,3760,64,1855,1219,4134


In [ ]:
from pathlib import Path

# Repo root (1 level up from notebooks/)
BASE_DIR = Path.cwd().parents[0]

output_dir = BASE_DIR / "results" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

file_path = output_dir / "status_quo_results.csv"
file_path

In [15]:
# Export to a CSV
status_quo_results.to_csv(file_path)